# Admin Demand Forecast
Multiple Linear Regression — Predict orders per hour (6 PM - 11 PM).

Run cells top-to-bottom: **Setup → Generate Data → Train → Evaluate → Test**

## 1. Setup
Resolve paths for `data/`, `output/`, `report/` directories.

In [ ]:
from pathlib import Path

try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path().resolve()

BASE_DIR   = NOTEBOOK_DIR.parent
DATA_DIR   = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
REPORT_DIR = BASE_DIR / "report"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR   : {BASE_DIR}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"REPORT_DIR : {REPORT_DIR}")

## 2. Generate Data
Create synthetic training data and save to `data/`.

In [ ]:
import csv
import random
import math
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
DATA_DIR = SCRIPT_DIR.parent / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)

def normal_noise(mu=0, sigma=1):
    u1 = random.random()
    u2 = random.random()
    z = math.sqrt(-2 * math.log(u1)) * math.cos(2 * math.pi * u2)
    return mu + sigma * z

hours = list(range(6))   # 0=6PM, 1=7PM, ..., 5=11PM
rows = []
for day in range(90):
    is_weekend = 1 if (day % 7) >= 5 else 0
    for h in hours:
        hour_sq = h * h
        orders = (5 + 40 * h - 7 * hour_sq + 15 * is_weekend + normal_noise(0, 3))
        orders = max(0, round(orders, 2))
        rows.append({
            "day": day,
            "hour_encoded": h,
            "hour_squared": hour_sq,
            "is_weekend": is_weekend,
            "orders": orders
        })

out_path = DATA_DIR / "demand_data.csv"
with open(out_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["day", "hour_encoded", "hour_squared", "is_weekend", "orders"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} rows to {out_path}")

## 3. Train
Fit the model and save to `output/`.

In [ ]:
import csv
import pickle
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
DATA_DIR = SCRIPT_DIR.parent / "data"
OUTPUT_DIR = SCRIPT_DIR.parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Pure Python matrix operations ----------
def _T(A):
    return [[A[j][i] for j in range(len(A))] for i in range(len(A[0]))]

def _mul(A, B):
    m, n, p = len(A), len(A[0]), len(B[0])
    return [[sum(A[i][k] * B[k][j] for k in range(n)) for j in range(p)] for i in range(m)]

def _inv(A):
    n = len(A)
    M = [A[i][:] + [1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    for col in range(n):
        pivot = max(range(col, n), key=lambda r: abs(M[r][col]))
        M[col], M[pivot] = M[pivot], M[col]
        M[col] = [x / M[col][col] for x in M[col]]
        for row in range(n):
            if row != col:
                f = M[row][col]
                M[row] = [M[row][k] - f * M[col][k] for k in range(2 * n)]
    return [M[i][n:] for i in range(n)]

def fit_lr(X, y):
    Xb = [[1] + row for row in X]
    Xt = _T(Xb)
    theta = _mul(_inv(_mul(Xt, Xb)), _mul(Xt, [[v] for v in y]))
    return [t[0] for t in theta]

def predict_lr(theta, X):
    return [sum(t * x for t, x in zip(theta, [1] + row)) for row in X]

# ---------- Load data ----------
rows = []
with open(DATA_DIR / "demand_data.csv", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

X_all = [[int(r["hour_encoded"]), int(r["hour_squared"]), int(r["is_weekend"])] for r in rows]
y_all = [float(r["orders"]) for r in rows]

split = int(0.8 * len(X_all))
X_train, X_test = X_all[:split], X_all[split:]
y_train, y_test = y_all[:split], y_all[split:]

theta = fit_lr(X_train, y_train)

model = {
    "theta": theta,
    "X_test": X_test,
    "y_test": y_test,
    "feature_names": ["intercept", "hour_encoded", "hour_squared", "is_weekend"]
}

out_path = OUTPUT_DIR / "demand_forecast_model.pkl"
with open(out_path, "wb") as f:
    pickle.dump(model, f)

print(f"Model saved to {out_path}")
print(f"Theta: {[round(t, 4) for t in theta]}")

## 4. Evaluate
Compute metrics on the test set and save to `report/`.

In [ ]:
import pickle
import json
import math
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
OUTPUT_DIR = SCRIPT_DIR.parent / "output"
REPORT_DIR = SCRIPT_DIR.parent / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def predict_lr(theta, X):
    return [sum(t * x for t, x in zip(theta, [1] + row)) for row in X]

def r2(y_true, y_pred):
    mean_y = sum(y_true) / len(y_true)
    ss_res = sum((t - p) ** 2 for t, p in zip(y_true, y_pred))
    ss_tot = sum((t - mean_y) ** 2 for t in y_true)
    return 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

def mae(y_true, y_pred):
    return sum(abs(t - p) for t, p in zip(y_true, y_pred)) / len(y_true)

def rmse(y_true, y_pred):
    return math.sqrt(sum((t - p) ** 2 for t, p in zip(y_true, y_pred)) / len(y_true))

with open(OUTPUT_DIR / "demand_forecast_model.pkl", "rb") as f:
    model = pickle.load(f)

theta = model["theta"]
X_test = model["X_test"]
y_test = model["y_test"]

y_pred = predict_lr(theta, X_test)

r2_score = r2(y_test, y_pred)
mae_score = mae(y_test, y_pred)
rmse_score = rmse(y_test, y_pred)

report = {
    "model": "Multiple Linear Regression",
    "task": "Admin hourly demand forecast",
    "train_samples": 432,
    "test_samples": 108,
    "r2": round(r2_score, 4),
    "mae": round(mae_score, 4),
    "rmse": round(rmse_score, 4)
}

out_path = REPORT_DIR / "evaluation_report.json"
with open(out_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Report saved to {out_path}")
print(f"R²={r2_score:.4f}  MAE={mae_score:.4f}  RMSE={rmse_score:.4f}")

## 5. Test
Run inference on sample inputs.

In [ ]:
import pickle
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
OUTPUT_DIR = SCRIPT_DIR.parent / "output"

def predict_lr(theta, X):
    return [sum(t * x for t, x in zip(theta, [1] + row)) for row in X]

with open(OUTPUT_DIR / "demand_forecast_model.pkl", "rb") as f:
    model = pickle.load(f)

theta = model["theta"]

hour_labels = ["6PM", "7PM", "8PM", "9PM", "10PM", "11PM"]

print("\n=== Admin Demand Forecast: Orders per Hour ===")
print(f"{'Hour':<6} {'Weekday':>10} {'Weekend':>10}")
for h in range(6):
    hs = h * h
    weekday_pred = predict_lr(theta, [[h, hs, 0]])[0]
    weekend_pred = predict_lr(theta, [[h, hs, 1]])[0]
    print(f"{hour_labels[h]:<6} {weekday_pred:>10.1f} {weekend_pred:>10.1f}")